[![](https://i.imgur.com/DJjibkb.png)](https://lambdalabs.com/mlb-kaggle)  

Hi all, this is one of many walkthroughs I'll be posting over this BDB season. If you like this notebook, please upvote and if you _really_ like this content check out my streams on Twitch, Monday - Wednesday starting at 8pm ET.

https://twitch.tv/nickwan_datasci  


# Simple Feature Engineering  
Since this year's competition is surrounding pre-snap to post-snap predicitions, pre- and post-snap features will be critical to develop in order to accurately assess pre-snap value. I have taken several features (speed, acceleration, distance) and created new metrics, both pre- and post-snap in order to better understand the relationship between pre-snap motion and post-snap distance covered.  

I start with loading in packages and setting my project directory path. Then I load in the first week of data, info about the plays, and some info about players (to get their primary positions). 

In [ ]:
import os 
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 
project_dir = '/kaggle/input/nfl-big-data-bowl-2025'

In [ ]:
plays = pd.read_csv(f'{project_dir}/plays.csv')
players = pd.read_csv(f'{project_dir}/players.csv')
df = pd.read_csv(f'{project_dir}/tracking_week_1.csv')

On my [stream](https://twitch.tv/nickwan_datasci), we randomly selected the following game featuring DEN and SEA. This should work with all games -- you can probably loop through all games or set up a function that allows you to get these features. 

In [ ]:
# DEN vs SEA, but you can use any game ID 
gid = 2022091200

The following code is all of the feature engineering (the code comments might be more helpful than my text here). I start by subsetting the data by the `gid` in order to work with less data. I also join info about the team with possession from `plays`. On my stream, there was some initial graphing comparing offense and defense. Not a necessary step but I found it instructive for any BDB beginners.  

Next, I summed distances from the player tracking dataframe, grouping by play-player-"frame type" (pre-snap / snap / post-snap). I also find max speed and acceleration in a similar manner. I merge all of this together into a frame called `stage` -- which I use in my code as a "staging" table in order to collect all the features I am engineering.  

I also wanted to know distance away from the ball on the Y axis at different times pre-snap. I merge the ball location at snap back onto the player tracking dataframe in order to use it to get relative player distances. For each player, I find their Y-location on frame 1 (which is a pre-snap frame) and also at snap (which is roughly the last frame before "post-snap"; you can be more precise and select snap-minus-one-frame if you'd like). Then, I subtract the ball Y-location from the "beginning" of the pre-snap phase and the "end" of the pre-snap phase -- and then finally, subtracting the beginning from the end to get a grand distance pre-snap. This can be helpful for contextualizing not just total distance pre-snap but if a player has "flipped" from one side of the field to the other (and/or how often they made these changes).  

I merge player positions onto the stage dataframe and make a new column that identifies secondary players from WR/TE/RB, leaving all other players labeled as neither secondary nor WR/TE/RB. This helps with some graphing aesthetics.  

In [ ]:
# subset one game 
data = df.loc[(df['gameId']==gid)].copy()

# merge the team with possession and tag explicitly 
_df = plays.loc[(plays['gameId']==gid), ['gameId', 'playId', 'possessionTeam']].rename(columns={'possessionTeam':'club'})
_df['posTeam'] = 1
data = data.merge(_df, how='left')
data['posTeam'] = data['posTeam'].fillna(0)

# create distance covered using the `dis` column
_df = data.loc[:, ['frameType','playId', 'posTeam', 'nflId', 'displayName', 'dis']].groupby(['frameType', 'playId', 'posTeam', 'nflId', 'displayName'], as_index=False).sum().rename(columns={'dis':'distCovered'})

# create speed and acc maxes 
_df2 = data.loc[:, ['frameType', 'playId', 'posTeam','nflId', 'displayName', 's', 'a']].groupby(['frameType', 'playId', 'posTeam', 'nflId', 'displayName'], as_index=False).max().rename(columns={'s':'sMax', 'a':'aMax'})

# merge everything together 
_df = _df.merge(_df2)

# widen the dataframe -- could pivot but this felt faster on stream  
_df = _df.loc[_df['frameType']=='BEFORE_SNAP'].merge(_df.loc[_df['frameType']=='AFTER_SNAP'], on=['playId', 'posTeam', 'nflId','displayName'], suffixes=('PreSnap','PostSnap'))

# drop some columns that we won't need 
stage = _df.drop(['frameTypePreSnap','frameTypePostSnap'],axis=1)

# join ball loc data for relative distance from ball 
_df = data.loc[(data['frameType']=='SNAP') & (data['displayName']=='football'), ['playId','x','y']].rename(columns={'x':'ballX', 'y':'ballY'})
stage = stage.merge(_df)

# I use frame 1 for the start of pre-snap and the snap frame for the end of pre-snap  
_df1 = data.loc[(data['frameId']==1), ['playId','nflId','y']].copy()
_df1 = _df1.rename(columns={'y':'pY1'})

_df2 = data.loc[(data['frameType']=='SNAP'), ['playId','nflId','y']].copy()
_df2 = _df2.rename(columns={'y':'pY2'})
_df = _df1.merge(_df2)
stage = stage.merge(_df)

# subtract ball y-position from player y-positions  
stage['yDistBallFrameS'] = stage['pY1'].sub(stage['ballY'])
stage['yDistBallFrameF'] = stage['pY2'].sub(stage['ballY'])
stage['yDistBallFrameD'] = stage['yDistBallFrameF'].sub(stage['yDistBallFrameS'])

# clear up cols  
stage = stage.drop(['ballX', 'ballY', 'pY1', 'pY2'], axis=1)
stage = stage.merge(players.loc[:, ['nflId', 'position']])

# make tags for graph purposes 
secondary = [
    'SS', 'CB','FS',
]
receiver = [
    'WR','RB', 'TE',
]
stage['pos'] = 'Neither'
stage.loc[stage['position'].isin(secondary), 'pos'] = 'Secondary'
stage.loc[stage['position'].isin(receiver), 'pos'] = 'WR/TE/RB'

After engineering some pre- and post-snap features, we could create some simple analyses to see how pre-snap behavior relates to post-snap behavior. You could create models based off of these features alone, or come up with more comprehensive features using similar methodology.  

Below are two graphs I found easy to explain:  
First, max speed pre-snap more clearly correlates with WR/TE/RB while players in the secondary and players who are neither secondary nor WR/TE/RB are less correlated. This makes some sense -- WR/TE/RB motion is driven by play design while defensive motion (many times) is reactionary to the offensive formation and motion. Furthermore, defensive secondary may not move pre-snap as WR/TE/RB does. Playing zone coverage or playing off the line of scrimmage may decrease the amount of pre-snap motion a defensive player may create.  
Second, I isolated only WR/TE/RB to see if there was an actual line that fit positively. If you squint, there seems to be a weak correlation. Some data clean up could help breathe a little more R2 into this correlation (e.g. pass plays only would probably help).  

In [ ]:
x = 'sMaxPreSnap'
y = 'distCoveredPostSnap'
sns.scatterplot(data=stage, x=x, y=y, hue='pos')
plt.title('All players, pre-snap speed * post-snap distance')
sns.despine()
plt.show()

sns.regplot(data=stage.loc[stage['pos']=='WR/TE/RB'], x=x, y=y)
plt.title('WR/TE/RB only, pre-snap speed * post-snap distance')
sns.despine()
plt.show()

# Streams on Twitch 
# Monday - Wednesday 
# 8 pm ET 
# https://twitch.tv/nickwan_datasci 
# nolook kissyy 

[![](https://i.imgur.com/DJjibkb.png)](https://lambdalabs.com/mlb-kaggle)  
Free $20 of credit on Lambda, click the Lambda logo above! 